# 策略概述

**HDBSCAN** 是命題 1 的分組層之一：以**機器學習密度式分群取代 GICS 靜態產業分類**
來建立配對搜尋空間。

作法是把每檔股票定位在一個 **19 維混合特徵空間**——它對共同風險因子的暴露（5 維報酬
PCA 因子載荷）、它的公司性質（2 維 PIT 基本面）、以及它的產業歸屬（12 維 GICS
one-hot）——讓演算法依**密度**自動把相近的股票分成一群，群內再找配對。

檢驗的問題是：**資料驅動的分群，能不能找到比靜態產業分類更高品質的配對？**

::: {.callout-note}

### 特徵矩陣為何與 Agglomerative、K-means 完全相同

命題 1 是 4 分組 × 3 排序的消融矩陣，**分組方法是唯一變因**。
三種分群法若各用各的特徵，量到的就是「特徵 × 演算法」的混合效果。
故 HDBSCAN、Agglomerative、K-means 共用同一份 19 維混合特徵矩陣
（`feature_mode="fundamentals_mix"`），差別只在分群演算法本身。

:::


# 輸入值卡

::: {.callout-important}

### 這個模型吃什麼、吐什麼

| 項目 | 內容 |
| :--- | :--- |
| **輸入特徵** | **19 維混合特徵**，三個區塊**各自** `StandardScaler` 後依權重拼接（不做 joint 標準化，否則 one-hot 的欄位數會主導尺度）：<br/>① **價格行為 5 維** — 形成窗 252 日日報酬矩陣 → 逐股標準化（等同對相關矩陣做 PCA）→ 前 5 個主成分，載荷以 $\sqrt{\text{特徵值}}$ 加權<br/>② **公司基本面 2 維** — $\log(1+\text{市值})$、盈餘殖利率 $1/PE$；產業中位數插補 + winsorize<br/>③ **GICS 產業 one-hot 12 維** — 11 個 GICS 產業 + 1 個「未分類」<br/>**連續維度共 7 維** |
| **輸出** | $\{$股票 $\to$ 群標籤$\}$。噪音點（`label = -1`）與過小群（< 5 檔）一併映射為 `"Unknown"`，不參與配對 |
| **超參數** | `pca_n_components` = 5、`hdbscan_min_cluster_size` = 5、`hdbscan_min_samples` = 2（值越大越保守、判為雜訊者越多）、`metric` = euclidean、三區塊權重各 1.0 |
| **標籤** | **無**。非監督式分群，只讀形成窗內的資料，**不使用未來報酬**，故分組層本身無前視風險 |
| **時點控制** | 基本面走 **Point-in-Time**：每個形成窗只取「日期 $\le$ 形成期末」的最近一筆，來源 `dataset/fundamental/sp500_pit_2000_2025_monthly.parquet` |
| **隨機性** | PCA 固定 `random_state`；HDBSCAN 為決定性演算法 → 同輸入必得同輸出 |

:::

::: {.callout-warning}

### 產業資訊經三條管道進入，不只 one-hot 一條

除了區塊③的明確 one-hot，**區塊②的缺失值插補**（填產業中位數 = 為缺資料股票加上
產業標籤）與**排序層的 SSD**（同業價格路徑天然更近）也各自帶入產業先驗。

僅改插補方式即可使跨產業配對比例自 15.7% 變到 59.3%——見第五章實證貢獻（其四）。

:::

::: {.callout-note}

### 檔名裡的 `pca5` 指的是價格區塊，不是整個特徵矩陣

本筆記本早期版本描述的是「5 維純報酬載荷」的 PCA5 設定（不含基本面與 one-hot）。
該設定已非現役——現役 36 條策略全部走 `cluster_formation` 且全部使用
`fundamentals_mix`；PCA5 的回測結果仍保留於 `result.db` 供追溯。

檔名維持不變以免既有投影片連結失效；`pca5` 現在僅指區塊①取 5 個主成分。

:::


# 策略架構

與 GICS 對照組**唯一的差異是「分組」層**——用 HDBSCAN 分群取代 GICS 產業；
特徵、排序、篩選、交易完全相同，確保比較是乾淨的單變因對照。

```{mermaid}
flowchart LR
  P["特徵<br/>19 維混合（報酬 PCA ⊕ 基本面 ⊕ 產業）"] --> G["分組<br/>HDBSCAN 密度式分群（取代 GICS）"]
  G --> F["篩選<br/>群內共整合 + 半衰期 + Hurst"]
  F --> R["排序<br/>SSD ＋ DTW 主成分融合"]
  R --> T["Top N 配對<br/>→ 交易期"]
```

| 層 | 本策略採用 | 用途 |
| :--- | :--- | :--- |
| 特徵 | 19 維混合（5 報酬 PCA ⊕ 2 基本面 ⊕ 12 產業 one-hot） | 三種分群法共用，使分組成為唯一變因 |
| 分組 | **HDBSCAN 密度式分群** | 資料驅動分群，取代 GICS |
| 排序 | SSD 與 DTW 主成分融合（`ssd_dtw_pca`） | 綜合兩種距離 |
| 篩選 | 群內共整合 + 半衰期 + Hurst | 確認價差均值回歸 |
| 交易 | Z-Score（標準化空間） | 偏離進場、回歸出場 |

**對應策略**：`Grid (HDB-SSD)`、`Grid (HDB-DTW)`、`Grid (HDB-SDP)`——
分別為三種排序準則下的 HDBSCAN 臂；`Grid (HDB-SDP)` 另有 DL-THR 交易端版本
`Grid (HDB-SDP-DRL)`。

> 排序與篩選的**施行次序依排序後端而異**（SSD 先排序後檢定、DTW 系先檢定後排序），
> 見第三章 §3.2。


## 特徵矩陣的三個區塊為何這樣配

### 為何價格區塊只取 5 維

報酬 PCA 的主成分依解釋變異量遞減排列：前少數主成分對應市場、規模、產業景氣等
**強共同因子**，後段主成分解釋變異低、載荷值以雜訊成分為主。

密度式分群以歐氏距離估計密度——特徵向量中每一維**等權**參與距離計算。
若保留過多低訊號維度，雜訊維度的隨機差異會稀釋強因子維度上的密度結構，
使群落邊界模糊：可分群的股票被誤判為噪音、群數不穩定。

取 $k = 5$ 使價格座標只由訊號最強的因子構成：

- 每檔股票的 5 維向量 $\approx$ 它在五大共同風險因子上的暴露組合
- 因子暴露相近的股票在此空間中自然聚攏
- 對每期形成窗的雜訊實現不敏感，滾動窗之間的分群結果更穩定

> 已封存的 **15 維版**正是反例：15 維空間平均每期僅解釋 58.6% 報酬變異，
> 後段主成分訊號弱卻等權參與距離計算，平均 30.9% 標的被判為雜訊、群數在 2~25 間震盪。

### 為何要加基本面與產業區塊

純報酬載荷只描述「一起漲跌」，不描述「為何一起漲跌」。
加入公司性質（規模、估值）與產業歸屬，使群落同時具備**行為相似**與**經濟相似**，
後者是共整合關係得以持續的基礎（Hong & Hwang, 2021）。

### 為何三區塊各自標準化

one-hot 區塊有 12 欄、連續區塊只有 7 欄。若對整個矩陣做 joint 標準化，
歐氏距離會被欄位數較多的 one-hot 主導。三區塊各自 `StandardScaler` 後再依權重
（現役皆為 1.0）拼接，使「一個區塊」而非「一個欄位」成為權重的單位。


# 參考文獻與引用對應


## 文獻 1：Avellaneda & Lee (2010)

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance, 10*(7), 761–782.

**參考部分**：

- 以 PCA 對報酬相關矩陣做特徵分解萃取共同風險因子（eigenportfolios），並指出**有效因子數量遠小於股票數量**：少數主成分即涵蓋市場的主要系統性變異
- 特徵向量以 $\sqrt{\text{特徵值}}$ 尺度化

**為何參考**：

- 「只取前少數強因子作為股票的風險表徵」即本策略取 $k=5$ 的理論基礎；其餘載荷計算方式與尺度化同前



## 文獻 2：Sarmento & Horta (2020)

> Sarmento, S. M., & Horta, N. (2020). Enhancing a pairs trading strategy with the application of machine learning. *Expert Systems with Applications, 158*, 113490.

**參考部分**：

- 「**降維** → 非監督式分群 → 規則化配對篩選」三段式框架——其中降維步驟的目的即為分群提供**低維、高訊號**的表徵

**為何參考**：

- 本策略把框架中的降維步驟落實為「PCA 取前 5 主成分」：分群品質依賴輸入表徵的訊噪比，降維是框架的必要成分而非可選項



## 文獻 3：Hong & Hwang (2021)

> Hong, S., & Hwang, S. (2021). Pairs trading with fundamental characteristics.
> *(公司特徵於配對搜尋空間之應用)*

**參考部分**：

- 以公司基本面特徵（規模、估值）而非僅價格路徑界定「相似股票」

**為何參考**：

- 特徵矩陣區塊②（$\log$ 市值、$1/PE$）的依據：純報酬相似只描述共動，
  基本面相似則提供共動得以**持續**的經濟解釋
- 與 Agglomerative 策略共用同一組基本面特徵，故兩者之間的差異純為分群演算法


## 文獻 4：Campello, Moulavi & Sander (2013)

> Campello, R. J., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD 2013*.

**參考部分**：

- HDBSCAN 演算法本體：層次密度估計、自動群數決定、噪音標記（`label = -1`）
- 密度估計對輸入空間距離度量的依賴性——距離中的雜訊維度直接影響密度結構的清晰度

**為何參考**：

- 自動群數決定與離群過濾同前；其密度估計原理亦為本策略「壓縮特徵維度以保全密度訊號」的依據


## 文獻 5：許鈞翔 (2025)／Engle & Granger (1987)／Krauss, Do & Huck (2016)

> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。元智大學碩士論文。
> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction. *Econometrica, 55*(2).
> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies. *EJOR*.

**參考部分與理由**：

- 許鈞翔 (2025)：群內配對的完整篩選與排序流程（Engle-Granger $p<0.01$ → SSD/DTW → PC1 融合排序）
- Engle & Granger：雙向 OLS + ADF 兩步驟共整合程序
- Krauss et al.：半衰期（$1$–$42$ 日）與 Hurst（$H<0.5$）門檻


# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動）內依序執行以下六個階段。


## 階段 1：19 維混合特徵矩陣建構

**區塊① 價格行為（5 維）**

1. 日報酬矩陣逐股標準化（PCA 等同對報酬**相關矩陣**做特徵分解）：

$$R \in \mathbb{R}^{(T-1) \times N}, \qquad R^{s}_{\cdot,i} = \frac{R_{\cdot,i} - \mu_i}{\sigma_i}$$

2. PCA 取 $k = \min(5,\ T-2,\ N-1)$ 個主成分
3. 每檔股票的特徵向量（$\sqrt{\text{特徵值}}$ 加權）：

$$\text{loadings}_i = \text{components}_{:,i} \times \sqrt{\text{explained variance}} \in \mathbb{R}^{5}$$

**區塊② 公司基本面（2 維）**

$\log(1 + \text{市值})$ 與盈餘殖利率 $1/PE$；缺失以**產業中位數插補**後 winsorize。
資料走 Point-in-Time：每個形成窗只取日期 $\le$ 形成期末的最近一筆。

**區塊③ GICS 產業 one-hot（12 維）**

11 個 GICS 產業各一欄，加上「未分類」一欄。

**拼接**

$$X = \big[\ \underbrace{\text{scale}(\text{PCA}) \cdot w_p}_{5}\ \big\|\
\underbrace{\text{scale}(\text{基本面}) \cdot w_f}_{2}\ \big\|\
\underbrace{\text{one-hot} \cdot w_s}_{12}\ \big] \in \mathbb{R}^{N \times 19}$$

現役 $w_p = w_f = w_s = 1.0$。各區塊**分別**標準化的理由見上節。


## 階段 2：HDBSCAN 密度式分群

對 $N \times 19$ 的混合特徵矩陣**直接**執行 HDBSCAN，不再降維
（區塊①已在特徵層完成降維）：

| 參數 | 值 |
| :--- | :---: |
| `hdbscan_min_cluster_size` | 5 |
| `hdbscan_min_samples` | 2 |
| `metric` | euclidean |

- 噪音點（`label = -1`）與**規模 < `min_cluster_size` = 5 的群**一併映射為
  `"Unknown"`，排序端自動跳過
- 群數由密度自動決定，不需事先指定——這是 HDBSCAN 與 Agglomerative／K-means
  的主要差異（後兩者的粒度由切割門檻或 $k$ 決定）


## 階段 3：群內雙向 OLS 與統計篩選

以群標籤（`Cluster_0`, `Cluster_1`, ...）作為分組，執行：

1. **雙向 OLS + ADF 方向決定**：兩個回歸方向各檢定一次，取 $p$ 值較小者
2. **三道統計篩選**：ADF $p < 0.05$ → OU 半衰期 $1 \le HL \le 42$ 日（$\lambda < 0$）
   → Hurst $H < 0.50$

> **門檻為 0.05 而非 0.01**：`_GRID_COMMON` 的 `adf_pvalue_threshold` = 0.05，
> 為消融矩陣全格共用值（0.01 是已封存 PCA5 設定的舊值）。
> 0.01／0.05／0.10 的敏感度對照見附錄 E。


## 階段 4：SSD 與 DTW 距離計算

$$\text{SSD}_{A,B} = \sum_{t=1}^{F} \left(P'_{A,t} - P'_{B,t}\right)^2$$

$$D(i,j) = (P'_{A,i} - P'_{B,j})^2 + \min\big\{ D(i-1,j),\ D(i,j-1),\ D(i-1,j-1) \big\}, \qquad |i-j| \le 15$$


## 階段 5：PCA 融合排序與配對選取

1. SSD 與 DTW 各自標準化 → PCA 取 PC1 分數（loadings 為負則取反）
2. 依 PC1 分數升序取前 `top_n` 組

輸出欄位與群／真實產業雙軌記錄（`Sector` = 群標籤；`Sector_A/B` = 真實 GICS 回填，
供交易期對配對兩腳各自計算產業曝險）。


## 階段 6：交易期的參數使用方式

`ignore_ols_alpha=True`，交易期於標準化空間重建 spread：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \qquad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio} \cdot P'_{B,t}, \qquad
Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

形成期統計量整個交易期凍結不變（無前視）。交易決策細節見 `trading/zscore_trading.ipynb`。
本策略的形成期配對另供 DL-THR 門檻選擇式交易端使用
（`Grid (HDB-SDP-DRL)`，見 `trading/drl_threshold_trading.ipynb`）。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗 / 滾動步長 | 252 / 21 交易日 | 全流程輸入 | 約一年 / 一個月 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依融合分數升序取前幾組 |
| **特徵模式** | `fundamentals_mix` | 特徵 | **19 維；三種分群法共用** |
| 價格因子數 | 5 | 特徵 | 區塊①維度（前 5 大共同因子） |
| 三區塊權重 | 各 1.0 | 特徵 | 價格 / 基本面 / 產業 等權 |
| 基本面資料源 | Point-in-Time 逐點 | 特徵 | 每窗取日期 ≤ 形成期末的最近一筆 |
| 最小群大小 | 5 | 分組 | 群至少要幾檔股票才成立 |
| 分群密度保守度 | 2 | 分組 | `hdbscan_min_samples`；值越大越保守、噪音越多 |
| 距離衡量 | SSD + DTW（主成分融合） | 排序 | `ranking_backend = ssd_dtw_pca` |
| DTW 頻帶寬 | 15 | 排序 | Sakoe-Chiba 窗寬 |
| 共整合顯著水準 | **0.05** | 篩選 | 消融矩陣全格共用（敏感度見附錄 E） |
| 半衰期 / Hurst | $[1,\ 42]$ 日 / <0.5 | 篩選 | 均值回歸過濾 |
| 隨機種子 | PCA 固定 | 特徵 | HDBSCAN 本身為決定性 |
